# CFPS submission reconciliation — PocketBase vs. GP-annotated export

**Purpose.** Compare an existing `clusters_designs.json` against the current authorized `cfps_designs` records in PocketBase. Produce a machine-readable discrepancy report and a safely refreshed input for downstream analysis **only if every existing submission has the same recipe and protein target**. This notebook never refits the Gaussian process, invents GP predictions, or treats a modified recipe as retaining valid annotations.

**Data governance:** read-only database access. All output files, including reports and refreshed JSON, may contain private participant identifiers and formulations. Do not commit or publish them. No credentials are written to outputs.

**How to run:** Open in Google Colab; upload the original `clusters_designs.json` in the upload cell; configure PocketBase credentials via Colab Secrets (`PB_TOKEN`, or `PB_EMAIL` and `PB_PASSWORD`); run all cells. Download the final private ZIP. Upload the report here for review before using any refreshed dataset.

**Safe behavior:** If a record is missing, new, or its reagent volumes/protein target differ, the notebook refuses to write a GP-ready refreshed export. It still writes a discrepancy report and an unannotated PocketBase snapshot suitable for the main notebook's `pocketbase_snapshot` mode. Exact matches can be re-exported without changing model-derived fields. IDs are never guessed or matched by approximate chemical similarity.

In [ ]:
%pip -q install requests nbformat

## Configuration

Use `PB_URL` matching your authorized database. The default below matches the main release notebook; verify it against your deployment. `PB_TOKEN` must be a **PocketBase** token, not a Keycloak or unrelated API token. Set `PB_AUTH_COLLECTION` to the collection permitted by your deployment.

In [ ]:
from pathlib import Path
import os, json, hashlib, zipfile, datetime, math, getpass
from collections import Counter
import requests

PB_URL = os.environ.get("PB_URL", "https://db.rcdonovan.com").rstrip("/")
PB_AUTH_COLLECTION = os.environ.get("PB_AUTH_COLLECTION", "_superusers")
COLLECTION = "cfps_designs"
PAGE_SIZE = 200
OUTPUT_DIR = Path("/content/cfps_reconciliation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# False means do not write a refreshed GP-annotated export when any ID, recipe or target differs.
STRICT_GP_REFRESH = True

## Load the GP-annotated export

In [ ]:
from google.colab import files
uploaded = files.upload()
assert "clusters_designs.json" in uploaded, "Upload the exact file named clusters_designs.json"
source_bytes = uploaded["clusters_designs.json"]
annotated = json.loads(source_bytes)
assert isinstance(annotated.get("proteins"), dict), "Expected top-level proteins mapping"
print("GP export SHA-256:", hashlib.sha256(source_bytes).hexdigest())

## Authenticate and fetch a complete, read-only PocketBase snapshot

In [ ]:
try:
    from google.colab import userdata
except ImportError:
    userdata = None

def secret(name):
    value = os.environ.get(name, "")
    if not value and userdata is not None:
        try: value = userdata.get(name) or ""
        except Exception: pass
    return value

token = secret("PB_TOKEN")
if not token:
    email = secret("PB_EMAIL") or input("Authorized PocketBase email: ")
    password = secret("PB_PASSWORD") or getpass.getpass("PocketBase password: ")
    resp = requests.post(f"{PB_URL}/api/collections/{PB_AUTH_COLLECTION}/auth-with-password",
                         json={"identity":email,"password":password}, timeout=30)
    resp.raise_for_status()
    token = resp.json()["token"]
session = requests.Session()
session.headers.update({"Authorization": token, "Accept": "application/json"})
records=[]
expected_total=None
page=1
while True:
    resp=session.get(f"{PB_URL}/api/collections/{COLLECTION}/records",
                     params={"page":page,"perPage":PAGE_SIZE,"sort":"id"}, timeout=60)
    resp.raise_for_status()
    result=resp.json()
    if expected_total is None: expected_total=int(result["totalItems"])
    batch=result["items"]
    records.extend(batch)
    if not batch or len(records)>=expected_total: break
    page+=1
assert len(records)==expected_total, f"Incomplete PocketBase fetch: {len(records)}/{expected_total}"
assert len({str(r['id']) for r in records})==len(records), "Duplicate PocketBase IDs"
snapshot={"metadata":{"created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "designs_advertised_total_items":expected_total,"designs_possibly_truncated":False,
    "source":"PocketBase/cfps_designs","base_url":PB_URL},"collections":{"cfps_designs":records}}
(OUTPUT_DIR/"pocketbase_snapshot.json").write_text(json.dumps(snapshot,indent=2,ensure_ascii=False))
print("Fetched",len(records),"records; snapshot saved privately")

## Validate and compare exact immutable IDs, target proteins, and dispensing volumes

All reagent quantities are interpreted as **dispensing volumes in nL**. Numeric differences within `1e-6` nL are ignored solely to accommodate JSON numeric formatting. Metadata and authorship are summarized separately; they do not invalidate a chemical match. No model-derived annotations are recalculated.

In [ ]:
TARGETS=("sfGFP","PETase","Reteplase")
def target(x):
    lookup={"sfgfp":"sfGFP","petase":"PETase","reteplase":"Reteplase"}
    k=str(x or "").lower().replace("-","").replace(" ","")
    if k not in lookup: raise ValueError(f"Unknown protein target {x!r}")
    return lookup[k]

def reagent_vector(x, label):
    if isinstance(x,str): x=json.loads(x)
    if not isinstance(x,dict) or not x: raise ValueError(f"{label}: missing recipe object")
    v={}
    for k,value in x.items():
        if isinstance(value,bool): raise ValueError(f"{label}: boolean volume")
        f=float(value)
        if not math.isfinite(f) or f<0: raise ValueError(f"{label}: invalid volume {k}")
        if f>0: v[str(k)]=f
    return v

def equivalent(a,b):
    return set(a)==set(b) and all(math.isclose(a[k],b[k],abs_tol=1e-6,rel_tol=0) for k in a)

old={}
for p in TARGETS:
    for d in annotated["proteins"][p]["designs"]:
        rid=str(d["id"])
        if rid in old: raise ValueError(f"Duplicate annotated ID: {rid}")
        old[rid]={"target":p,"recipe":reagent_vector(d["recipeNl"],rid),"design":d}

new={}
excluded=Counter()
for r in records:
    payload=r.get("design_json")
    if isinstance(payload,str): payload=json.loads(payload)
    if not isinstance(payload,dict): raise ValueError(f"Invalid design_json for {r['id']}")
    kind=str(payload.get("sample_type","")).lower().strip()
    if kind!="experimental":
        excluded[kind or "missing"]+=1
        continue
    metadata=payload.get("metadata") or {}
    if isinstance(metadata,str): metadata=json.loads(metadata)
    rid=str(r["id"])
    if rid in new: raise ValueError(f"Duplicate experimental ID: {rid}")
    new[rid]={"target":target(metadata.get("protein_target")),
              "recipe":reagent_vector(payload.get("reagent_list"),rid),"record":r}

old_ids=set(old); new_ids=set(new)
missing=sorted(old_ids-new_ids)
added=sorted(new_ids-old_ids)
changed=[]; exact=[]
for rid in sorted(old_ids & new_ids):
    a,b=old[rid],new[rid]
    if a["target"]!=b["target"] or not equivalent(a["recipe"],b["recipe"]):
        changed.append({"id":rid,"annotated_target":a["target"],"pocketbase_target":b["target"],
            "annotated_recipeNl":a["recipe"],"pocketbase_recipeNl":b["recipe"]})
    else: exact.append(rid)

report={"schema":"cfps-gp-export-reconciliation/v1",
 "source_sha256":hashlib.sha256(source_bytes).hexdigest(),
 "pocketbase_snapshot_sha256":hashlib.sha256((OUTPUT_DIR/"pocketbase_snapshot.json").read_bytes()).hexdigest(),
 "counts":{"annotated_experimental":len(old),"pocketbase_experimental":len(new),
           "exact_matches":len(exact),"missing_from_pocketbase":len(missing),
           "new_in_pocketbase":len(added),"changed_recipe_or_target":len(changed)},
 "excluded_nonexperimental_types":dict(excluded),
 "missing_from_pocketbase_ids":missing,"new_in_pocketbase_ids":added,
 "changed_records":changed,
 "gp_refresh_safe":not (missing or added or changed),
 "warning":"Recipe or target changes invalidate existing GP forecasts and cluster/run annotations; no GP refit performed."}
(OUTPUT_DIR/"reconciliation_report.json").write_text(json.dumps(report,indent=2,ensure_ascii=False,allow_nan=False))
print(json.dumps(report["counts"],indent=2))
print("GP-annotated export safe to re-use:",report["gp_refresh_safe"])

## Write a GP-compatible verified copy only on an exact match

This step is intentionally conservative. It does not transplant model forecasts onto new or changed formulations. A complete match produces a byte-for-byte copy of the annotated JSON (not a newly trained GP export), plus a reconciliation report. When differences exist, use the unannotated `pocketbase_snapshot.json` with `INPUT_SOURCE = "pocketbase_snapshot"` in the main notebook, or regenerate GP annotations with the original modeling pipeline before running GP-informed modes.

In [ ]:
if report["gp_refresh_safe"]:
    path=OUTPUT_DIR/"clusters_designs_verified.json"
    path.write_bytes(source_bytes)
    print("Verified copy:",path)
else:
    print("GP-compatible export withheld: discrepancies detected. See reconciliation_report.json")
    print("The PocketBase snapshot can be analyzed in model_free mode after validation.")

## Download private reconciliation artifacts

In [ ]:
zip_path=Path("/content/cfps_reconciliation_private.zip")
manifest={}
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.glob("*.json")):
        manifest[path.name]=hashlib.sha256(path.read_bytes()).hexdigest()
        z.write(path,arcname=path.name)
    z.writestr("SHA256_MANIFEST.json",json.dumps(manifest,indent=2))
print("Private archive:",zip_path,"; files:",list(manifest))
from google.colab import files
files.download(str(zip_path))